In [ ]:
# ===============================================================================
# CELL 1: INSTALLATION, FUNCTIONS, AND SINGLE EXECUTION EXAMPLE (ADAPTED FOR AAE)
# ===============================================================================
!pip install --quiet google-api-python-client pandas

import os
import time
import pandas as pd
from google.colab import drive
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import nltk
from nltk.corpus import stopwords

In [ ]:
# --- NLTK Downloads (run only once) ---
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('taggers/averaged_perceptron_tagger')
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('averaged_perceptron_tagger', quiet=True)
    nltk.download('stopwords', quiet=True)

In [ ]:
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
# --- Mount Google Drive ---
try:
    drive.mount('/content/drive')
except:
    print("Drive already mounted or error mounting.")

In [ ]:
# ==============================================================================
# EXTRACTION FUNCTIONS AND API QUERY (No changes to internal logic)
# ==============================================================================

def extract_key_terms(text):
    """Extracts nouns, verbs, and adjectives from a text, removing stopwords."""
    if not isinstance(text, str):
        return []
    tokens = nltk.word_tokenize(text.lower())
    english_stopwords = set(stopwords.words('english'))
    tagged_words = nltk.pos_tag(tokens)
    key_terms = [word for word, tag in tagged_words if tag.startswith(('NN', 'VB', 'JJ')) and word not in english_stopwords]
    return list(set(key_terms))

def get_google_kg_definitions(query_terms, service):
    """Searches for definitions for a list of terms in the Google Knowledge Graph."""
    if not query_terms:
        return 'No key concepts extracted.'
    definitions = []
    for term in query_terms:
        try:
            request = service.entities().search(query=term, languages='en', limit=1)
            response = request.execute()
            if 'itemListElement' in response and response['itemListElement']:
                result = response['itemListElement'][0]['result']
                name = result.get('name', 'N/A')
                final_description = "No description available."
                if 'detailedDescription' in result:
                    final_description = result['detailedDescription'].get('articleBody', "No detailed description text.")
                elif 'description' in result:
                    final_description = result.get('description')
                definitions.append(f"Query: '{term}' -> Result: '{name}'. Definition: {final_description}")
        except Exception:
            continue
    if definitions:
        return ' '.join(list(set(definitions)))
    else:
        return 'No relevant definitions found for the key concepts.'

# ==============================================================================
# SINGLE CASE TEST WITH THE AAE_TEST.CSV DATASET
# ==============================================================================

# --- Configuration ---
API_KEY = "" # Your API key here
DATASET_PATH = '/content/drive/My Drive/ProjectFolder/aae_test.csv' # Generic path

print(f"--- Running test with the first row of the dataset '{os.path.basename(DATASET_PATH)}' ---")

if not API_KEY or API_KEY == "YOUR_API_KEY_HERE":
    print("⚠️ WARNING: Google API key not configured.")
else:
    try:
        df_test = pd.read_csv(DATASET_PATH)
        example_row = df_test.iloc[0]

        argument_text = example_row['argument']
        topic_text = example_row['topic']
        question_text = f'The argument "{argument_text}" SUPPORT or ATTACK the belief "{topic_text}"?'

        argument_entities = extract_key_terms(argument_text)
        topic_entities = extract_key_terms(topic_text)
        all_entities = list(set(argument_entities + topic_entities))

        print(f"\nArgument: {argument_text}")
        print(f"Topic: {topic_text}")
        print(f"Extracted key concepts: {all_entities}")

        kg_service = build('kgsearch', 'v1', developerKey=API_KEY, cache_discovery=False)
        start_time = time.time()
        knowledge_context = get_google_kg_definitions(all_entities, kg_service)
        end_time = time.time()

        print(f"\n⏱️ Query execution time: {end_time - start_time:.4f} seconds")
        print("\n--- Generated Prompt for the Example ---")

        final_prompt_example = (
            f"Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' "
            f"which shows direct paths between concepts from the argument and the belief. "
            f"Based on this, decide if the argument attacks or supports the belief. "
            f"Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.\n\n"
            f"Knowledge Graph Context: {knowledge_context}\n\n"
            f"Question: {question_text}"
        )
        print(final_prompt_example)

    except FileNotFoundError:
        print(f"❌ ERROR: File not found at: {DATASET_PATH}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

In [ ]:
# ==============================================================================
# CELL 2: MAIN PROCESSING SCRIPT (ADAPTED FOR AAE)
# ==============================================================================
from tqdm.auto import tqdm

# --- Configuration ---
API_KEY = "" # Your API key here
INPUT_CSV_PATH = '/content/drive/My Drive/ProjectFolder/aae_test.csv' # Generic path
OUTPUT_CSV_PATH = '/content/drive/My Drive/ProjectFolder/aae_test_prompts.csv' # Output file

if not API_KEY or API_KEY == "YOUR_API_KEY_HERE":
    print("⚠️ WARNING: Google API key not configured. The script cannot continue.")
else:
    # --- 1. DATA LOADING ---
    try:
        df = pd.read_csv(INPUT_CSV_PATH)
        print(f"File '{os.path.basename(INPUT_CSV_PATH)}' loaded successfully. Total of {len(df)} rows.")
    except FileNotFoundError:
        print(f"❌ ERROR: File not found at the specified path: {INPUT_CSV_PATH}")
        df = None
    except Exception as e:
        print(f"An unexpected error occurred during loading: {e}")
        df = None

    if df is not None:
        # --- 2. BATCH PROCESSING ---
        results_data = []
        kg_service = build('kgsearch', 'v1', developerKey=API_KEY, cache_discovery=False)

        print(f"\nStarting prompt generation for the {len(df)} samples in the dataset.")
        for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Generating prompts"):

            question_prompt = (
                f"The argument \"{row['argument']}\" "
                f"SUPPORT or ATTACK the belief \"{row['topic']}\"?"
            )

            # Extract key concepts and search for context
            argument_entities = extract_key_terms(row['argument'])
            topic_entities = extract_key_terms(row['topic'])
            all_entities = list(set(argument_entities + topic_entities))
            knowledge_context = get_google_kg_definitions(all_entities, kg_service)

            # Build the final prompt
            final_prompt = (
                f"Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' "
                f"which shows direct paths between concepts from the argument and the belief. "
                f"Based on this, decide if the argument attacks or supports the belief. "
                f"Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.\n\n"
                f"Knowledge Graph Context: {knowledge_context}\n\n"
                f"Question: {question_prompt}"
            )

            # Add results to the list, keeping the original columns and adding the new ones
            results_data.append({
                'ID': row['ID'],
                'SET': row['SET'],
                'topic': row['topic'],
                'argument': row['argument'],
                'knowledge_context': knowledge_context,
                'final_prompt': final_prompt,
                'ground_truth_stance': row['classification'] # Mapping 'classification' to the standard name
            })
            time.sleep(0.1) # Pause to avoid overloading the API

        # --- 3. SAVING THE RESULTS ---
        results_df = pd.DataFrame(results_data)
        results_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8')
        print(f"\n✅ Process completed! The file '{os.path.basename(OUTPUT_CSV_PATH)}' was saved successfully.")
        display(results_df.head())

In [ ]:
results_df.ground_truth_stance.value_counts()

In [ ]:
print(results_df.final_prompt[0])